# Cruce de Graduados USTA con **SECOP Integrado**

**Objetivo.** Identificar cuáles de las cédulas de los graduados (dataset consolidado
en `salidas/graduados_integrado.csv`) aparecen como **proveedores/contratistas** del
Estado en **SECOP Integrado**, y caracterizar su contratación (número de contratos,
valor total, periodo).

## Fuente: SECOP Integrado (Datos Abiertos Colombia)

- Conjunto de datos: **SECOP Integrado** — `rpmr-utcd`
  (https://www.datos.gov.co/Estad-sticas-Nacionales/SECOP-Integrado/rpmr-utcd).
- Se consulta vía **API Socrata (SoQL)**, el mismo origen usado en el ejercicio de
  clase (`https://www.datos.gov.co/resource/rpmr-utcd.csv`).
- Tamaño: **~22 millones** de contratos; el documento del contratista está en la
  columna **`documento_proveedor`** (para personas naturales es la cédula, en texto).

## Estrategia (eficiente, sin descargar 22M de filas)

Tenemos ~174 685 cédulas únicas. En lugar de bajar todo SECOP, se hace el cruce
**del lado del servidor**: por lotes se envía una consulta
`documento_proveedor IN (lista_de_cedulas)` agrupada por documento, de modo que la
API **solo devuelve las coincidencias** ya agregadas (n.º de contratos, valor total,
nombre y fechas). Así el tráfico es proporcional a los *match*, no a los 22M.

> **Criterio de match:** igualdad exacta del número de documento. La cédula es el
> identificador nacional único, por lo que un match indica que esa misma persona
> figura como contratista. (Limitaciones al final.)


## 1. Configuración e importaciones

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

API_URL   = "https://www.datos.gov.co/resource/rpmr-utcd.json"  # SECOP Integrado
COL_DOC   = "documento_proveedor"
TAM_LOTE  = 250          # cédulas por petición (IN-list)
PAUSA     = 0.10         # segundos entre peticiones (cortesía con la API)
TIMEOUT   = 120

# Token de aplicación opcional de Socrata. Sin token funciona, pero con límites más
# bajos; si tienes uno, ponlo aquí para subir el cupo de peticiones.
APP_TOKEN = None
SESSION = requests.Session()
if APP_TOKEN:
    SESSION.headers.update({"X-App-Token": APP_TOKEN})

SALIDA = Path("salidas"); SALIDA.mkdir(exist_ok=True)
print("API:", API_URL)


## 2. Cédulas a consultar

Se cargan las identificaciones (solo dígitos) del dataset integrado y se toman las
**únicas y válidas** (numéricas, longitud razonable de documento colombiano).


In [ ]:
g = pd.read_csv(SALIDA / "graduados_integrado.csv",
                usecols=["identificacion", "fuente"], dtype={"identificacion": "string"})

ced = (g["identificacion"]
       .dropna()
       .str.strip())
ced = ced[ced.str.fullmatch(r"\d{4,12}")]      # documentos plausibles
cedulas_unicas = sorted(ced.unique())
print(f"Registros con identificación: {len(ced):,}")
print(f"Cédulas únicas a consultar  : {len(cedulas_unicas):,}")


## 3. Función de consulta por lotes (SoQL)

Cada petición agrupa por `documento_proveedor` y devuelve los agregados de
contratación. Incluye reintentos con *backoff* ante errores transitorios de red/API.


In [ ]:
SELECT = (
    "documento_proveedor,"
    "count(1) as n_contratos,"
    "sum(valor_contrato) as valor_total,"
    "max(nom_raz_social_contratista) as nombre_secop,"
    "max(tipo_documento_proveedor) as tipo_doc_secop,"
    "min(fecha_de_firma_del_contrato) as primera_firma,"
    "max(fecha_de_firma_del_contrato) as ultima_firma"
)

def consultar_lote(cedulas, intentos=4):
    '''Consulta un lote de cédulas en SECOP Integrado. Devuelve lista de dicts
    (una fila por cédula que SÍ aparece como proveedor).'''
    in_list = ",".join("'%s'" % c for c in cedulas)
    params = {
        "$select": SELECT,
        "$where": f"{COL_DOC} in ({in_list})",
        "$group": COL_DOC,
        "$limit": 50000,
    }
    for k in range(intentos):
        try:
            r = SESSION.get(API_URL, params=params, timeout=TIMEOUT)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if k == intentos - 1:
                raise
            time.sleep(2 ** k)        # backoff: 1s, 2s, 4s
    return []

def lotes(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


## 4. Ejecución del cruce

Recorre todas las cédulas en lotes. Es la parte que consume red (~250 peticiones).
Se reporta el progreso y se acumulan las coincidencias.


In [ ]:
# --- Caché-primero: si ya existe el dataset de proveedores, se omite la consulta a la API ---
_cache_prov = SALIDA / "graduados_proveedores_secop.csv"
USAR_CACHE = _cache_prov.exists()
if USAR_CACHE:
    prov = pd.read_csv(_cache_prov, dtype={"identificacion": "string"})[
        ["identificacion", "n_contratos", "valor_total", "nombre_secop",
         "tipo_doc_secop", "primera_firma", "ultima_firma"]].copy()
    print(f"Cache encontrada ({_cache_prov.name}): {len(prov):,} proveedores. Se omite la consulta a la API.")
    resultados = None
else:
    resultados = []
    total_lotes = (len(cedulas_unicas) + TAM_LOTE - 1) // TAM_LOTE
    t0 = time.time()
    for i, lote in enumerate(lotes(cedulas_unicas, TAM_LOTE), start=1):
        filas = consultar_lote(lote)
        resultados.extend(filas)
        if i % 20 == 0 or i == total_lotes:
            print(f"Lote {i:>3}/{total_lotes} | coincidencias acumuladas: {len(resultados):>5} | {time.time()-t0:5.0f}s")
        time.sleep(PAUSA)
    print(f"\nListo. Proveedores encontrados: {len(resultados):,} en {time.time()-t0:.0f}s")


## 5. Construcción de la tabla de proveedores

Se tipan los agregados y se ordena por valor total contratado.


In [ ]:
if resultados is not None:                      # los datos vinieron de la API
    prov = pd.DataFrame(resultados)
    if prov.empty:
        raise RuntimeError("No se obtuvieron resultados; revisa la conexión o la API.")
    prov = prov.rename(columns={"documento_proveedor": "identificacion"})

prov["identificacion"] = prov["identificacion"].astype("string")
prov["n_contratos"]    = pd.to_numeric(prov["n_contratos"], errors="coerce").astype("Int64")
prov["valor_total"]    = pd.to_numeric(prov["valor_total"], errors="coerce")
prov["primera_firma"]  = pd.to_datetime(prov["primera_firma"], errors="coerce")
prov["ultima_firma"]   = pd.to_datetime(prov["ultima_firma"], errors="coerce")
prov = prov.drop_duplicates("identificacion").sort_values("valor_total", ascending=False)

print("Proveedores únicos:", prov["identificacion"].nunique())
prov.head(10)


## 6. Marcar y enriquecer el dataset de graduados

Se hace `left join` del dataset integrado completo con los proveedores hallados,
creando la bandera `es_proveedor_secop`.


In [ ]:
grad = pd.read_csv(SALIDA / "graduados_integrado.csv", dtype={"identificacion": "string"})
grad["identificacion"] = grad["identificacion"].str.strip()

cruce = grad.merge(prov, on="identificacion", how="left")
cruce["es_proveedor_secop"] = cruce["n_contratos"].notna()

n_reg   = len(cruce)
n_prov_reg = int(cruce["es_proveedor_secop"].sum())
ced_total  = cruce["identificacion"].dropna().nunique()
ced_prov   = cruce.loc[cruce["es_proveedor_secop"], "identificacion"].nunique()

print(f"Registros de graduados            : {n_reg:,}")
print(f"  -> registros que son proveedores: {n_prov_reg:,} ({n_prov_reg/n_reg*100:.1f}%)")
print(f"Cédulas únicas de graduados       : {ced_total:,}")
print(f"  -> cédulas que son proveedores  : {ced_prov:,} ({ced_prov/ced_total*100:.1f}%)")


## 7. Análisis

### 7.1 Proporción de graduados-proveedores por fuente / programa


In [ ]:
tmp = cruce.dropna(subset=["identificacion"]).copy()
graduados_x_fuente   = tmp.groupby("fuente")["identificacion"].nunique()
proveedores_x_fuente = (tmp[tmp["es_proveedor_secop"]]
                        .groupby("fuente")["identificacion"].nunique())
por_fuente = pd.DataFrame({
    "graduados": graduados_x_fuente,
    "proveedores": proveedores_x_fuente,
}).fillna(0).astype(int)
por_fuente["pct_proveedores"] = (por_fuente["proveedores"] / por_fuente["graduados"] * 100).round(1)
display(por_fuente)

ax = por_fuente["pct_proveedores"].plot(kind="bar", color="#1B998B")
ax.set_title("% de graduados que son proveedores en SECOP, por fuente")
ax.set_ylabel("% proveedores"); ax.set_xlabel("Fuente")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


In [ ]:
# Top programas por número de graduados-proveedores (cédulas únicas)
prov_prog = (cruce[cruce["es_proveedor_secop"]]
             .dropna(subset=["identificacion"])
             .groupby("programa_norm")["identificacion"].nunique()
             .sort_values(ascending=False).head(15))
ax = prov_prog.sort_values().plot(kind="barh", color="#274690")
ax.set_title("Top 15 programas por n.º de graduados que son proveedores en SECOP")
ax.set_xlabel("Cédulas únicas proveedoras"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
prov_prog.to_frame("proveedores")


### 7.2 Magnitud de la contratación de los graduados-proveedores

In [ ]:
print(f"Valor total contratado (suma)         : ${prov['valor_total'].sum():,.0f}")
print(f"Contratos totales                     : {int(prov['n_contratos'].sum()):,}")
print(f"Mediana de valor por proveedor        : ${prov['valor_total'].median():,.0f}")
print(f"Mediana de contratos por proveedor    : {prov['n_contratos'].median():.0f}")
print("\nTop 15 graduados-proveedores por valor total contratado:")
top = prov.head(15)[["identificacion", "nombre_secop", "n_contratos", "valor_total", "ultima_firma"]].copy()
top["valor_total"] = top["valor_total"].map(lambda v: f"${v:,.0f}")
display(top)


## 7.3 Setup del análisis ampliado

A partir de aquí enriquecemos el cruce básico con tres dimensiones adicionales que se
extraen de las tablas externas: el **departamento** de la entidad contratante, las
**características del contrato** (tipo, modalidad, nivel de la entidad) y el **objeto**
del contrato según el clasificador **UNSPSC** (este último desde *SECOP II*, pues el
SECOP Integrado no lo expone).

Cada extracción usa el patrón **cargar-o-extraer**: si el CSV ya existe en `salidas/` se
carga; si no, se consulta la API por lotes y se guarda. Así el notebook es reproducible
sin repetir descargas largas.

In [ ]:
import time, re, unicodedata
from pathlib import Path
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

SAL = Path("salidas")
API_INTEGRADO = "https://www.datos.gov.co/resource/rpmr-utcd.json"  # SECOP Integrado
API_SECOP2    = "https://www.datos.gov.co/resource/jbjy-vk9h.json"  # SECOP II (trae UNSPSC)
SES = requests.Session()

gi = pd.read_csv(SAL / "graduados_integrado.csv", dtype={"identificacion": "string"})
_ced = gi["identificacion"].dropna().str.strip()
CEDULAS = sorted(_ced[_ced.str.fullmatch(r"\d{4,12}")].unique())
# cédula -> programa principal y seccional
ced_prog = gi.dropna(subset=["identificacion"]).groupby("identificacion")["programa_norm"].agg(lambda s: s.value_counts().index[0])
ced_sec  = gi.dropna(subset=["identificacion"]).groupby("identificacion")["fuente"].first()
SECC = ["General", "Tunja", "Villavicencio"]
SECC_LBL = {"General": "Bucaramanga", "Tunja": "Tunja", "Villavicencio": "Villavicencio"}

def _na(s):
    s = unicodedata.normalize("NFKD", str(s))
    return "".join(c for c in s if not unicodedata.combining(c))

def consultar_lotes(api, select, group, tam=250, intentos=5):
    '''Consulta agregada por lotes de CEDULAS contra una API Socrata. Devuelve DataFrame.'''
    res = []
    for i in range(0, len(CEDULAS), tam):
        lote = CEDULAS[i:i+tam]
        inlist = ",".join("'%s'" % c for c in lote)
        params = {"$select": select, "$where": f"documento_proveedor in ({inlist})",
                  "$group": group, "$limit": 50000}
        for k in range(intentos):
            try:
                r = SES.get(api, params=params, timeout=180); r.raise_for_status()
                res.extend(r.json()); break
            except Exception:
                if k == intentos - 1: raise
                time.sleep(2 ** k)
        time.sleep(0.05)
    return pd.DataFrame(res)

print("Cédulas a consultar:", f"{len(CEDULAS):,}")

## 7.4 Dimensión territorial: ¿dónde contratan?

Se recupera el `departamento_entidad` de cada contrato (agregado por cédula y
departamento) y se cruza con la seccional de origen del graduado.

In [ ]:
f = SAL / "secop_regional.csv"
if f.exists():
    sreg = pd.read_csv(f, dtype={"identificacion": "string"})
    print("Cargado de caché:", f.name)
else:
    sreg = consultar_lotes(API_INTEGRADO,
        "documento_proveedor, departamento_entidad, count(1) as n_contratos, sum(valor_contrato) as valor",
        "documento_proveedor, departamento_entidad")
    sreg = sreg.rename(columns={"documento_proveedor": "identificacion", "departamento_entidad": "departamento"})
    sreg["n_contratos"] = pd.to_numeric(sreg["n_contratos"], errors="coerce")
    sreg["valor"] = pd.to_numeric(sreg["valor"], errors="coerce")
    sreg.to_csv(f, index=False, encoding="utf-8-sig")
    print("Extraído y guardado:", f.name)

sreg["dep"] = sreg["departamento"].map(lambda s: _na(s).strip()).replace({"Distrito Capital de Bogota": "Bogota D.C."})
dep = sreg.groupby("dep")["n_contratos"].sum().sort_values(ascending=False)
dep = dep[dep.index != "(sin dato)"]
print("Filas:", f"{len(sreg):,}")
ax = dep.head(12).sort_values().plot(kind="barh", color="#274690", figsize=(7,4))
ax.set_title("Departamentos donde contratan los graduados (SECOP)"); ax.set_xlabel("Contratos")
plt.tight_layout(); plt.show()
dep.head(12).to_frame("contratos")

In [ ]:
# Departamento del contrato por seccional (% por fila): arraigo regional
sreg["seccional"] = sreg["identificacion"].map(ced_sec)
topd = dep.head(7).index.tolist()
piv = (sreg[sreg["seccional"].isin(SECC)]
       .assign(d=lambda x: x["dep"].where(x["dep"].isin(topd), "Otros"))
       .pivot_table(index="seccional", columns="d", values="n_contratos", aggfunc="sum", fill_value=0)
       .reindex(SECC))
pivp = (piv.div(piv.sum(axis=1), axis=0) * 100).round(1)
pivp.index = [SECC_LBL[s] for s in SECC]
print("Cada seccional contrata sobre todo en su propio departamento:")
pivp

## 7.5 Características del contrato: tipo, modalidad y nivel de la entidad

Se extraen, por cédula, el `tipo_de_contrato`, la `modalidad_de_contratacion` y el
`nivel_entidad`. Las categorías se consolidan (mayúsculas/tildes) para evitar duplicados.

In [ ]:
f = SAL / "secop_dims.csv"
if f.exists():
    dims = pd.read_csv(f, dtype={"identificacion": "string"})
    print("Cargado de caché:", f.name)
else:
    dims = consultar_lotes(API_INTEGRADO,
        "documento_proveedor, tipo_de_contrato, modalidad_de_contrataci_n, nivel_entidad, count(1) as n_contratos, sum(valor_contrato) as valor",
        "documento_proveedor, tipo_de_contrato, modalidad_de_contrataci_n, nivel_entidad", tam=200)
    dims = dims.rename(columns={"documento_proveedor": "identificacion", "modalidad_de_contrataci_n": "modalidad"})
    dims["n_contratos"] = pd.to_numeric(dims["n_contratos"], errors="coerce")
    dims["valor"] = pd.to_numeric(dims["valor"], errors="coerce")
    dims.to_csv(f, index=False, encoding="utf-8-sig")
    print("Extraído y guardado:", f.name)

for c in ["tipo_de_contrato", "modalidad", "nivel_entidad"]:
    dims[c] = dims[c].fillna("(sin dato)").str.strip().str.title()
dims["nivel_entidad"] = dims["nivel_entidad"].replace({"Corporación Autónoma": "Territorial", "No Definido": "(sin dato)"})
print("Filas:", f"{len(dims):,}")

In [ ]:
# Tipo de contrato
t = (dims.groupby("tipo_de_contrato")
     .agg(proveedores=("identificacion", "nunique"), contratos=("n_contratos", "sum"), valor=("valor", "sum"))
     .sort_values("contratos", ascending=False))
t["%contratos"] = (t["contratos"] / t["contratos"].sum() * 100).round(1)
print("La prestación de servicios domina abrumadoramente:")
display(t.head(8))

# Nivel de la entidad: contratos vs valor
nv = dims.groupby("nivel_entidad").agg(contratos=("n_contratos", "sum"), valor=("valor", "sum"))
nv["%contratos"] = (nv["contratos"] / nv["contratos"].sum() * 100).round(1)
nv["%valor"] = (nv["valor"] / nv["valor"].sum() * 100).round(1)
print("\nEl nivel territorial concentra los contratos; el nacional, el valor:")
display(nv)

## 7.6 Objeto del contrato: clasificador UNSPSC (SECOP II)

El SECOP Integrado no expone el código UNSPSC del objeto contratado; sí lo hace **SECOP II**
(`jbjy-vk9h`). Se cruzan las cédulas con SECOP II (cobertura parcial: contratos
electrónicos) para recuperar `codigo_de_categoria_principal`, se mapea a categorías de
objeto y se revela la **firma disciplinar** de cada carrera al apartar el genérico de
apoyo profesional.

In [ ]:
f = SAL / "secop_unspsc.csv"
if f.exists():
    uns = pd.read_csv(f, dtype={"identificacion": "string"})
    print("Cargado de caché:", f.name)
else:
    uns = consultar_lotes(API_SECOP2,
        "documento_proveedor, codigo_de_categoria_principal, count(1) as n, sum(valor_del_contrato) as valor",
        "documento_proveedor, codigo_de_categoria_principal", tam=300)
    uns = uns.rename(columns={"documento_proveedor": "identificacion", "codigo_de_categoria_principal": "unspsc"})
    uns["n"] = pd.to_numeric(uns["n"], errors="coerce")
    uns["valor"] = pd.to_numeric(uns["valor"], errors="coerce")
    uns.to_csv(f, index=False, encoding="utf-8-sig")
    print("Extraído y guardado:", f.name)

uns["code"] = uns["unspsc"].astype(str).str.extract(r"(\d{8})")[0]

def categoria_unspsc(code):
    '''Mapea un código UNSPSC de 8 dígitos a una categoría de objeto legible.'''
    if not isinstance(code, str) or len(code) < 2:
        return "Otros"
    seg, fam = code[:2], code[:4]
    if fam == "8012": return "Jurídico"
    if seg in ("93", "94", "92"): return "Cívico y político"
    if seg in ("85", "42"): return "Salud"
    if seg == "81": return "Ingeniería y TI"
    if seg in ("72", "95", "30", "31"): return "Construcción y obra"
    if seg == "84": return "Financiero y contable"
    if seg == "77": return "Ambiental"
    if seg == "86": return "Educación"
    if seg in ("70", "71"): return "Agro y recursos"
    if seg == "78": return "Transporte"
    if seg == "80": return "Apoyo profesional y gestión"
    return "Otros"

uns["cat"] = uns["code"].map(categoria_unspsc)
uns["prog"] = uns["identificacion"].map(ced_prog)
catall = uns.groupby("cat")["n"].sum().sort_values(ascending=False)
generico = round(catall.get("Apoyo profesional y gestión", 0) / catall.sum() * 100, 1)
print(f"Contratos en SECOP II: {int(uns['n'].sum()):,} | categoría genérica de apoyo: {generico}%")
catall.to_frame("contratos")

In [ ]:
# Firma disciplinar: carrera x objeto ESPECIALIZADO (excluido el apoyo genérico)
ESPEC = ["Jurídico", "Salud", "Ingeniería y TI", "Construcción y obra",
         "Financiero y contable", "Cívico y político", "Ambiental", "Educación"]
topprogs = uns["prog"].value_counts().head(8).index.tolist()
he = uns[uns["prog"].isin(topprogs) & uns["cat"].isin(ESPEC)]
pv = he.pivot_table(index="prog", columns="cat", values="n", aggfunc="sum", fill_value=0).reindex(topprogs)
cols = [c for c in ESPEC if c in pv.columns]; pv = pv[cols]
pvp = (pv.div(pv.sum(axis=1).replace(0, np.nan), axis=0) * 100).round(0)

fig, ax = plt.subplots(figsize=(9, 4.2))
im = ax.imshow(pvp.values, cmap="Blues", aspect="auto")
ax.set_yticks(range(len(topprogs))); ax.set_yticklabels([str(p).title()[:24] for p in topprogs], fontsize=8)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=30, ha="right", fontsize=8)
for i in range(len(topprogs)):
    for j in range(len(cols)):
        v = pvp.values[i, j]
        if v == v: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7,
                           color="white" if v > 50 else "black")
fig.colorbar(im, label="% (entre contratos especializados)")
ax.set_title("Firma disciplinar: carrera × objeto especializado (UNSPSC)")
plt.tight_layout(); plt.show()
print("Cada profesión deja su firma: Odontología→Salud, Contaduría→Financiero, Ingenierías→Ingeniería, Derecho→Jurídico.")
pvp

## 7.7 Datasets ampliados generados

El análisis anterior deja en `salidas/` tres insumos reutilizables, además de los del
cruce básico:

- `secop_regional.csv` — departamento de la entidad por cédula (contratos y valor).
- `secop_dims.csv` — tipo de contrato, modalidad y nivel de la entidad por cédula.
- `secop_unspsc.csv` — código UNSPSC del objeto (desde SECOP II) por cédula.

## 8. Exportación

- `graduados_secop_cruce.csv`: dataset integrado completo + bandera y métricas SECOP.
- `graduados_proveedores_secop.csv`: solo las cédulas que son proveedores (con agregados).


In [ ]:
ruta_cruce = SALIDA / "graduados_secop_cruce.csv"
ruta_prov  = SALIDA / "graduados_proveedores_secop.csv"

cruce.to_csv(ruta_cruce, index=False, encoding="utf-8-sig")

# Proveedores con datos académicos (un registro por cédula proveedora; conserva el
# primer programa/sede asociado en nuestras bases).
prov_export = (cruce[cruce["es_proveedor_secop"]]
               .sort_values("valor_total", ascending=False)
               .drop_duplicates("identificacion")
               [["identificacion", "nombre_completo", "fuente", "sede", "programa",
                 "nombre_secop", "tipo_doc_secop", "n_contratos", "valor_total",
                 "primera_firma", "ultima_firma"]])
prov_export.to_csv(ruta_prov, index=False, encoding="utf-8-sig")

print("Exportado:")
print(" -", ruta_cruce.resolve())
print(" -", ruta_prov.resolve(), f"({len(prov_export):,} proveedores)")


## 9. Conclusiones y limitaciones

**Resultado.** Se cruzaron las cédulas de los graduados contra los ~22M de contratos
de SECOP Integrado y se marcó quiénes figuran como contratistas del Estado, con su
número de contratos, valor total y periodo de actividad.

**Limitaciones / advertencias metodológicas**
- **Match por número de documento.** Se asume que el `documento_proveedor` (cédula) en
  SECOP corresponde a la misma persona graduada. La cédula es única, pero pueden existir
  errores de digitación en SECOP.
- **NIT de persona natural.** Algunos contratistas naturales se registran como
  *Nit de Persona Natural* (cédula + dígito de verificación, p.ej. `12345678-9`). El
  match exacto por cédula **no** captura ese formato; quedaría como mejora extraer el DV.
- **Homónimos / reúso de cédula:** no se valida el nombre; el cruce es estrictamente por
  documento. Se exporta `nombre_secop` para permitir verificación manual.
- **Cobertura temporal de SECOP:** depende de lo publicado en Datos Abiertos; contratos
  muy antiguos o de regímenes especiales pueden no estar.
